In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
os.environ["LOKY_MAX_CPU_COUNT"] = "4"


In [2]:
df= pd.read_csv(r"C:\Users\sohai\Downloads\UAC_Predictive_Forecasting\HHS_Unaccompanied_Alien_Children_Program.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sohai\\Downloads\\UAC_Predictive_Forecasting\\HHS_Unaccompanied_Alien_Children_Program.csv'

In [ ]:
df.columns
print(df.columns.tolist())

# Starting ke 5 records
print("\nFirst 5 Records:")
display(df.head())

# Last ke 5 records
print("\nLast 5 Records:")
display(df.tail())

In [ ]:
print('rows,col:',df.shape)
print('\ncolumns:')
print(df.columns.tolist())
print('\nNum duplicates:',df.duplicated().sum())
print('\nMissing values :')
print(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
for col in df.columns:
    if df[col].nunique() >0:
        print(f'\n{col} value counts:')
        print(df[col].value_counts())

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
date_index = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
observed_dates = pd.DatetimeIndex(df["Date"].drop_duplicates())
missing_dates = date_index.difference(observed_dates)

print("Observed reporting dates:", len(observed_dates))
print("Calendar days in range:", len(date_index))
print("Missing calendar dates:", len(missing_dates))
print("Largest date gap (days):", df["Date"].diff().dt.days.max())

gap_summary = df[["Date"]].copy()
gap_summary["days_since_previous_report"] = gap_summary["Date"].diff().dt.days
display(gap_summary["days_since_previous_report"].value_counts().sort_index().to_frame("number_of_gaps"))

In [ ]:
df = df.set_index("Date").sort_index()
daily = df.asfreq("D")

stock_cols = ["Children in CBP custody", "Children in HHS Care"]
flow_cols = [
    "Children apprehended and placed in CBP custody*",
    "Children transferred out of CBP custody",
    "Children discharged from HHS Care"
]
all_numeric_cols = stock_cols + flow_cols

# Track which dates were originally observed before imputation.
daily["is_observed_reporting_date"] = daily.index.isin(observed_dates)

# Time interpolation is an explicit modelling choice, not a claim that these were actual reported values.
for col in all_numeric_cols:
    daily[col] = pd.to_numeric(daily[col], errors="coerce").interpolate(method="time", limit_direction="both")

daily[all_numeric_cols] = daily[all_numeric_cols].clip(lower=0)
print("Daily modelling table:", daily.shape)
display(daily.head())

### find duplicate rows

### find Missing values

### find to check Date range & historical data

### Exploratory Data Analysis (EDA)